# Fake News Detection - End-to-End NLP Project

Team: Sabeur, Philippe, Joao  
Duration: 3 days

We're building a small classifier that reads a news headline and decides if it's fake (0) or real (1). The whole thing lives in this one notebook so anyone on the team can open it and follow along.

---

## Who does what

| Member | Role | What they hand off |
|--------|------|---------------------|
| Sabeur | Data & preprocessing | A clean dataset and a `clean_text()` function |
| Philippe | Features & modelling | A trained `model_pipeline` |
| Joao | Evaluation & predictions | Final metrics and `submission.csv` |

### How the 3 days are split

**Day 1 - getting the data ready (Sabeur, with Philippe joining at the end)**
- Morning: load the data, look around, fix the obvious issues, do a bit of EDA
- Afternoon: write and test the preprocessing pipeline
- End of day: Philippe sets up a TF-IDF + baseline model just to sanity-check the cleaned data

**Day 2 - modelling (Philippe, then Joao)**
- Morning: train Logistic Regression, Naive Bayes and SVM on TF-IDF features
- Afternoon: tune hyperparameters with cross-validation, pick the best one
- End of day: Joao wires up the train/val split and the evaluation helpers

**Day 3 - testing and wrap-up (Joao, then everyone)**
- Morning: run the trained model on `test.csv` and generate predictions
- Early afternoon: error analysis - look at what we got wrong and why
- Late afternoon: review together, finalise the submission, prep the slides

### A few rules so we don't step on each other

- Sabeur's `clean_text()` is the only preprocessing function. Philippe and Joao reuse it as-is.
- The vectorizer and model live inside one sklearn `Pipeline` that Philippe builds. Joao reuses that exact object - no retraining.
- On test data we only call `.transform()` and `.predict()`. Never `.fit()` - that would be data leakage.

### The pipeline at a glance

```
raw csv  ->  clean  ->  preprocess  ->  vectorise  ->  train  ->  evaluate  ->  predict
             Sabeur     Sabeur          Philippe       Philippe   Joao         Joao
```

---
## Setup

All the imports we share across the notebook. If you need something specific to your section, add it inside that section.

In [ ]:
import re
import string

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)

import joblib

# Run these once, then comment them out
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('punkt')

RANDOM_STATE = 42
pd.set_option('display.max_colwidth', 120)

---
# 1. The data

Two files in `dataset/`:

- `training_data_lowercase.csv` - has `label` and `headline`. Labels are 0 (fake) or 1 (real). This is what we train on.
- `testing_data_lowercase_nolabels.csv` - just `headline`. No labels - this is what we have to predict.

Some examples to get a feel for it:

Training:
- 0 - *donald trump sends out embarrassing...*
- 1 - *pope francis just called out donald trump...*

Test:
- *germany's fdp look to fill schaeuble's big shoes...*
- *france's macron says his job not 'cool' cites talks with turkey's erdogan...*

---
# 2. Loading and cleaning the data - Sabeur

The plan here is simple: get the CSVs into pandas, sanity-check them, drop anything broken, and make sure the types are what we expect. By the end of this section we should have `df_train` and `df_test` in good shape.

In [ ]:
TRAIN_PATH = 'dataset/training_data_lowercase.csv'
TEST_PATH = 'dataset/testing_data_lowercase_nolabels.csv'

# The files are tab-separated and have no header row
df_train = pd.read_csv(TRAIN_PATH, sep='\t', header=None, names=['label', 'headline'])
df_test = pd.read_csv(TEST_PATH, sep='\t', header=None, names=['headline'])

print('Train shape:', df_train.shape)
print('Test  shape:', df_test.shape)
df_train.head()

In [ ]:
print('Missing in train:')
print(df_train.isna().sum())
print('\nMissing in test:')
print(df_test.isna().sum())

n_dup = df_train.duplicated().sum()
print(f'\nDuplicate rows in train: {n_dup}')
df_train = df_train.drop_duplicates().reset_index(drop=True)

df_train['label'] = df_train['label'].astype(int)
df_train['headline'] = df_train['headline'].astype(str)
df_test['headline'] = df_test['headline'].astype(str)

df_train.info()

## 2.1 A quick look at the data (EDA) - Sabeur

Before we touch any models we want to know what we're dealing with: how balanced are the classes, how long are the headlines, what words show up most. If something looks weird now, better to catch it before Philippe and Joao build on top.

In [ ]:
# How balanced are the two classes?
ax = df_train['label'].value_counts().plot(kind='bar', color=['#e74c3c', '#2ecc71'])
ax.set_xticklabels(['Fake (0)', 'Real (1)'], rotation=0)
ax.set_title('Class distribution')
plt.show()

# How long are the headlines, and does it differ by class?
df_train['n_words'] = df_train['headline'].str.split().str.len()
sns.histplot(data=df_train, x='n_words', hue='label', bins=40, kde=True)
plt.title('Headline length (words) by class')
plt.show()

---
# 3. Preprocessing - Sabeur

The goal here is one function, `clean_text()`, that everyone uses. Whatever happens to the text - lowercasing, tokenising, dropping stopwords, lemmatising - happens inside this function so the train and test data go through exactly the same pipeline.

What it does:
1. Lowercase everything (the data is already lowercase but we do it anyway, just in case).
2. Strip out punctuation and digits.
3. Tokenise (`"this is great"` -> `["this", "is", "great"]`).
4. Drop English stopwords and very short tokens.
5. Lemmatise so `running -> run`, `better -> good` (we picked lemmatisation over stemming because it reads better).

After running it on both DataFrames we get a `processed_text` column ready for vectorisation.

In [ ]:
STOPWORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()
PUNCT_RE = re.compile(f'[{re.escape(string.punctuation)}0-9]')


def clean_text(text: str) -> str:
    """Lowercase, strip punctuation/digits, drop stopwords, lemmatise."""
    text = text.lower()
    text = PUNCT_RE.sub(' ', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens]
    return ' '.join(tokens)


# Quick check on a noisy example
clean_text("Donald Trump sends out EMBARRASSING tweets!!! #fakenews")

In [ ]:
df_train['processed_text'] = df_train['headline'].apply(clean_text)
df_test['processed_text'] = df_test['headline'].apply(clean_text)

df_train[['headline', 'processed_text', 'label']].head()

---
# 4. Feature engineering - Philippe

Now we turn `processed_text` into numbers a model can chew on. A few options on the table:

- **Bag of Words** with `CountVectorizer` - just word counts.
- **TF-IDF** with `TfidfVectorizer` - same idea but downweights words that appear everywhere. Good baseline.
- **n-grams** - throw bigrams in too, sometimes helps catch short phrases.
- **Embeddings** (Word2Vec, GloVe, BERT) - heavier; only if we have time.

One thing to keep in mind: fit the vectorizer **only on the training fold**, then `.transform()` validation and test. Otherwise we're cheating.

In [ ]:
# Philippe - add your code here


---
# 5. Training the models - Philippe

We try a few classic text classifiers and pick whichever does best on the validation set:

- Logistic Regression - the usual strong baseline for text.
- Multinomial Naive Bayes - fast, plays nicely with sparse counts.
- Linear SVM - often the winner on TF-IDF.
- (Maybe Random Forest or XGBoost if there's time.)

How we go about it:
1. Stratified train/validation split.
2. Vectorizer + classifier inside a single `sklearn.Pipeline` so there's no leakage.
3. Tune with `GridSearchCV`.
4. Save the winning pipeline with `joblib.dump` so Joao can pick it up.

End result: one fitted `model_pipeline` object that takes a list of cleaned headlines and returns predictions.

In [ ]:
# Philippe - add your code here


---
# 6. Evaluation - Joao

Time to see how well the model actually does and where it slips up.

Metrics we care about:
- Accuracy - decent overall summary, but only if classes are balanced.
- Precision - of the headlines we said are real, how many actually are?
- Recall - of the real headlines out there, how many did we catch?
- F1 - the precision/recall combo, usually the most honest one-number summary.
- Confusion matrix - shows where the wrong calls are happening.

After the numbers come out, we eyeball the misclassified headlines. Often there's a pattern (very short headlines, certain topics, etc.) and that feedback helps Sabeur or Philippe tweak earlier stages.

In [ ]:
# Joao - add your code here


---
# 7. Predicting on the test set - Joao

The hard work is done - here we just plug the test data through the same pipeline.

A few things to be careful about:
- Use Sabeur's `clean_text()`. Don't reinvent it.
- Use Philippe's already-fitted `model_pipeline`. Don't refit it on test.
- On the test data: `.transform()` and `.predict()` only. Never `.fit()`.

Steps:
1. Load `testing_data_lowercase_nolabels.csv`.
2. Run `clean_text()` on every headline.
3. `predictions = model_pipeline.predict(processed_test)`.

In [ ]:
# Joao - add your code here


---
# 8. Final output - Joao

What we hand in at the end:
1. `submission.csv` - one column called `prediction`, with 0 or 1 for each test headline, in the original order.
2. `model_pipeline.joblib` - the saved pipeline, in case anyone wants to reproduce the predictions.
3. A short evaluation write-up: the metrics, the confusion matrix, and a few sentences about the most common error patterns we found.

In [ ]:
# Joao - add your code here


---
# 9. Wrap-up and ideas for later

What we ended up with: a small, reproducible pipeline that cleans headlines, turns them into TF-IDF features, runs them through a classifier, and spits out predictions for the unlabelled test set.

Things we could try if we had more time:
- Word embeddings (Word2Vec, GloVe) instead of TF-IDF.
- A pretrained transformer (BERT or RoBERTa) for the heavy-lifting baseline.
- A tiny Streamlit app so anyone can paste a headline and get a verdict.
- Tracking how accuracy holds up over time or across topics.

Teamwork makes the model work.